# BATCH EXPLANATIONS

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
import matplotlib.pyplot as plt
import decord as de
import random
from mmaction.apis import inference_recognizer, init_recognizer
from skimage.color import gray2rgb
from scipy import ndimage
import cv2
import csv
import shutil
import os
from tqdm import tqdm
import time
import torch
from mmcv import Config
from mmcv.parallel import collate, scatter
from mmaction.datasets.pipelines import Compose
from functools import partial
from mmaction.datasets import PIPELINES as PIPELINES_MMACTION
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(1, os.path.join(sys.path[0], '../video_xai'))
sys.path.insert(2, os.path.join(sys.path[0], '../video_xai/other_methods'))
from xai import *
from utils import *
from pipelines import *
from segmenters import *
from visualizers import *
from other_methods import SaliencyTubes, GradCAM, AOSA, EP

In [4]:
# Where to store segmentations
root_path = "Z:/Xavi/VideoXAI/batch_explanations"
videos_path = ""

models = ["TimeSformer", "TPN", "TANet"]
datasets = ["EtriActivity3D", "Kinetics400", "UCF101"]
datasets_folder = {
    'EtriActivity3D': 'Z:/Xavi/EtriActivity3D',
    'Kinetics400': 'Z:/Xavi/Kinetics400',
    'UCF101': 'Z:/Xavi/UCF101'}
datasets_list = {
    'EtriActivity3D': 'Z:/Xavi/EtriActivity3D/EtriActivity3D_1_test_video.txt',
    'Kinetics400': 'Z:/Xavi/Kinetics400/Kinetics400_test_video.txt', 
    'UCF101': 'Z:/Xavi/UCF101/UCF101_1_test_video.txt'}

# Weights used are from the first CV split training
weights = {dataset: {model: datasets_folder[dataset] + '/training/' + model + '/1/epoch_10.pth' for model in models} for dataset in datasets}

# Weights used for Kinetics400 are from the pre-trained models
root = 'C:/Users/Xavi/OneDrive - Universitat de les Illes Balears/Doctorat/Video explanations paper/TFM'
weights['Kinetics400'] = {
            'TimeSformer': root+'/weights/kinetics400/timesformer_divST_8x32x1_15e_kinetics400_rgb-3f8e5d03.pth',
            'TANet': root+'/weights/kinetics400/tanet_r50_dense_1x1x8_100e_kinetics400_rgb_20210219-032c8e94.pth',
            'TPN': root+'/weights/kinetics400/tpn_imagenet_pretrained_slowonly_r50_8x8x1_150e_kinetics_rgb-44362b55.pth'
            }

# For configs, we use the same as for EtriActivity3D
configs = {
            'TimeSformer': root+'/configs/EtriActivity3D/timesformer_divST_8x32x1_15e_kinetics400_rgb.py',
            'TANet': root+'/configs/EtriActivity3D/tanet_r50_dense_1x1x8_100e_kinetics400_rgb.py',
            'TPN': root+'/configs/EtriActivity3D/tpn_imagenet_pretrained_slowonly_r50_8x8x1_150e_kinetics_rgb.py'
            }

n_classes = {
    'EtriActivity3D': 55,
    'Kinetics400': 400,
    'UCF101': 101
}

N_VIDEOS = 30

pipelines = [PIPELINES.LIME_EXP_NEW, PIPELINES.KERNEL_SHAP_EXP_NEW, PIPELINES.LOCO_EXP_NEW, PIPELINES.UNIVARIATE_EXP_NEW, PIPELINES.RISE_EXP_NEW, PIPELINES.OCCLUSION_EXP_NEW]
# pipelines = [PIPELINES.OCCLUSION_EXP_NEW2]
# pipelines = [PIPELINES.LIME_EXP, PIPELINES.KERNEL_SHAP_EXP, PIPELINES.LOCO_EXP, PIPELINES.UNIVARIATE_EXP]
# pipelines = [PIPELINES.OCCLUSION_EXP]
# pipelines = [PIPELINES.LIME_EXP, PIPELINES.LOCO_EXP, PIPELINES.UNIVARIATE_EXP, PIPELINES.RISE_EXP, PIPELINES.OCCLUSION_EXP]
# pipelines = [PIPELINES.KERNEL_SHAP_EXP]
# pipelines_names = ['3D-Sampled-Occl-Sens']
# pipelines_names = ['3D-LIME', '3D-Kernel-SHAP', 'LV-LOCO', 'LV-Univ-Pred', '3D-RISE', '3D-Sampled-Occl-Sens']
pipelines_names = ['3D-LIME-NEW', '3D-Kernel-SHAP-NEW', 'LV-LOCO-NEW', 'LV-Univ-Pred-NEW', '3D-RISE-NEW', '3D-Sampled-Occl-Sens-NEW']
# pipelines_names = ['3D-Sampled-Occl-Sens-NEW2']
# pipelines_names = ['3D-LIME', '3D-Kernel-SHAP', 'LV-LOCO', 'LV-Univ-Pred']
# pipelines_names = ['3D-LIME', 'LV-LOCO', 'LV-Univ-Pred', '3D-RISE', '3D-Sampled-Occl-Sens']
# pipelines_names = ['3D-Kernel-SHAP']
# slic_methods = ['3D-LIME', '3D-Kernel-SHAP', 'LV-LOCO', 'LV-Univ-Pred']
slic_methods = ['3D-LIME-NEW', '3D-Kernel-SHAP-NEW', 'LV-LOCO-NEW', 'LV-Univ-Pred-NEW']

In [5]:
# # Inference function
# def batch_predict(videos, verbose=False, topk=5, read_folder=None):
#     results = []
#     for video in videos:
#         results.append(inference_recognizer(model, video))
#     return np.array(results)

def load_list(list_path):
    with open(list_path, mode='r') as infile:
        reader = csv.reader(infile, delimiter=' ')
        return [(row[0], row[1]) for row in reader]

# Load the configuration file and modify some parameters
def prepare_config(cfg, pre_trained_model, dataset_name, dataset_dir, logs_dir, n_classes, k):

    # Modify dataset type and path
    cfg.dataset_type = 'VideoDataset'
    cfg.data_root = dataset_dir + '/videos'
    cfg.data_root_val =  dataset_dir + '/videos'
    cfg.ann_file_train =  dataset_dir + '/' + dataset_name + '_' + str(k) + '_train_video.txt'
    cfg.ann_file_val =  dataset_dir + '/' + dataset_name  + '_' + str(k) + '_test_video.txt'
    cfg.ann_file_test =  dataset_dir + '/' + dataset_name + '_' + str(k) + '_test_video.txt'

    cfg.data.train.type = 'VideoDataset'
    cfg.data.train.ann_file = dataset_dir + '/' + dataset_name + '_' + str(k) + '_train_video.txt'
    cfg.data.train.data_prefix = cfg.data_root

    cfg.data.val.type = 'VideoDataset'
    cfg.data.val.ann_file =dataset_dir + '/' + dataset_name + '_' + str(k) + '_test_video.txt'
    cfg.data.val.data_prefix = cfg.data_root

    cfg.data.test.type = 'VideoDataset'
    cfg.data.test.ann_file = dataset_dir + '/' + dataset_name + '_' + str(k) + '_test_video.txt'
    cfg.data.test.data_prefix = cfg.data_root

    # The flag is used to determine whether it is omnisource training
    cfg.setdefault('omnisource', False)
    # Modify num classes of the model in cls_head
    cfg.model.cls_head.num_classes = n_classes
    # We can use the pre-trained TSN model
    cfg.load_from = pre_trained_model

    # Set up working dir to save files and logs.
    cfg.work_dir = logs_dir

    # The original learning rate (LR) is set for 8-GPU training.
    # We divide it by 8 since we only use one GPU.
    cfg.data.videos_per_gpu = 2
    cfg.data.workers_per_gpu = 4
    cfg.optimizer.lr = cfg.optimizer.lr / 8
    cfg.total_epochs = 10

    # We can set the checkpoint saving interval to reduce the storage cost
    cfg.checkpoint_config.interval = 10
    # We can set the log print interval to reduce the times of printing log
    cfg.log_config.interval = 5

    # Set seed thus the results are more reproducible
    cfg.seed = 777
    # set_random_seed(777, deterministic=False)
    cfg.gpu_ids = range(1)

    # Save the best
    cfg.evaluation.save_best=None

def sample_frames(video, pipeline):
    data = dict(
        total_frames=video.shape[0],
        label=-1,
        start_index=0,
        array=video,
        modality='RGB')
    p_video = pipeline(data)

    # Sort the frames, storing the indexes to later unsort them
    sorted_indices, sorted_frames = zip(*sorted([(i, f) for i, f in enumerate(p_video['frame_inds'])], key=lambda x: x[1]))

    # Get the frames from the video in the sorted order. Used for perturbation
    frames_in = video[sorted_frames,...]

    return sorted_indices, frames_in

def unsort_frames(sorted_indices, frames_in):
    unsorted_indices, _ = zip(*sorted(zip(range(frames_in.shape[0]), sorted_indices), key=lambda x: x[1]))
    frames_out = frames_in[unsorted_indices,...]
    return frames_out

# Inference function
def batch_predict(videos, sorted_indices):
    results = []
    for video in videos:
        frames_out = unsort_frames(sorted_indices, video)
        results.append(inference_recognizer(model, frames_out))
    return np.array(results)

@PIPELINES_MMACTION.register_module()
class CustomTransform:

    def __init__(self, clip_len=1, frame_interval=1, num_clips=1):
        self.clip_len = clip_len
        self.frame_interval = frame_interval
        self.num_clips = num_clips

    def __call__(self, results):

        array = results['array']
        imgs = [array[f,...] for f in range(array.shape[0])]
        results['imgs'] = imgs
        results['original_shape'] = imgs[0].shape[:2]
        results['img_shape'] = imgs[0].shape[:2]

        results['clip_len'] = self.clip_len
        results['frame_interval'] = self.frame_interval
        results['num_clips'] = self.num_clips
        return results

    def __repr__(self):
        return f'{self.__class__.__name__}()'

## Choose videos to explain

Pick N videos per dataset randomly and copy them to a folder. These will be the videos explained.

In [6]:
# chosen_datasets = ['EtriActivity3D', 'UCF101', 'Kinetics400']
chosen_datasets = ['UCF101']

UCF101_24_CLASSES = [7, 8, 10, 21, 22, 25, 27, 29, 32, 41, 43, 50, 67, 74, 76, 79, 80, 81, 83, 87, 91, 93, 96, 97]

In [7]:
videos_class = []
videos_path = []
progress = tqdm(range(N_VIDEOS*len(datasets)))


# Create folder
if not os.path.exists(root_path):
    os.mkdir(root_path)

for dataset in chosen_datasets:

    # Create dataset folder
    if not os.path.exists(os.path.join(root_path, dataset)):
        os.mkdir(os.path.join(root_path, dataset))
    
    # Create videos folder
    if not os.path.exists(os.path.join(root_path, dataset, 'videos')):
        os.mkdir(os.path.join(root_path, dataset, 'videos'))
    
    # Create small videos folder
    if not os.path.exists(os.path.join(root_path, dataset, 'videos_small')):
        os.mkdir(os.path.join(root_path, dataset, 'videos_small'))
    
    # Load dataset root folder, and list of videos with classes
    dataset_folder = datasets_folder[dataset]
    dataset_list = load_list(datasets_list[dataset])
    
    # Shuffle list
    random.shuffle(dataset_list)
    
    # Copy N_VIDEOS first elements of shuffled list
    # Create class folders when needed
    saved_videos = 0
    video_i = 0
    while saved_videos < N_VIDEOS:
        
        # Next video
        video_name, video_class = dataset_list[video_i]

        # Skip videos not in UCF101_24_CLASSES
        if dataset == 'UCF101' and int(video_class) not in UCF101_24_CLASSES:
            video_i += 1
            continue

        video = load_video(os.path.join(dataset_folder, 'videos', video_name))
        scale_factor = 256 / min(video.shape[1:3])
        video = resize_video(video, scale_factor, scale_factor)
        video = center_crop_video(video, 256)
        
        # Predict with all models. Only videos predicted correctly by the three models will be stored
        good = True
        for model_name in models:
            
            # Load model and make prediction
            cfg = Config.fromfile(configs[model_name])
            # print(weights[dataset][model_name])
            # print(dataset_folder[dataset])
            # print(n_classes[dataset])
            prepare_config(cfg, weights[dataset][model_name], dataset, dataset_folder, None, n_classes[dataset], 1)
            # model = build_model(cfg.model, train_cfg=cfg.get('train_cfg'), cfg=config_file.get('test_cfg'))
            model = init_recognizer(cfg, weights[dataset][model_name], device='cuda:0')
            pred_class = np.argmax(batch_predict([video])[0, ...])
            
            # If prediction is not correct, skip to next video
            if pred_class != int(video_class):
                good = False
                break
        
        # Save video
        if good:
            if not os.path.exists(os.path.join(root_path, dataset, 'videos', video_class)):
                os.mkdir(os.path.join(root_path, dataset, 'videos', video_class))
            if not os.path.exists(os.path.join(root_path, dataset, 'videos_small', video_class)):
                os.mkdir(os.path.join(root_path, dataset, 'videos_small', video_class))
            shutil.copyfile(os.path.join(dataset_folder, 'videos', video_name), os.path.join(root_path, dataset, 'videos', video_class, video_name))
            save_video(video, os.path.join(os.path.join(root_path, dataset, 'videos_small', video_class, video_name)))

            saved_videos += 1
            progress.update(1)
            
        video_i += 1

A Jupyter Widget

Config:
checkpoint_config = dict(interval=10)
log_config = dict(interval=5, hooks=[dict(type='TextLoggerHook')])
dist_params = dict(backend='nccl')
log_level = 'INFO'
load_from = 'Z:/Xavi/UCF101/training/TimeSformer/1/epoch_10.pth'
resume_from = None
workflow = [('train', 1)]
opencv_num_threads = 0
mp_start_method = 'fork'
model = dict(
    type='Recognizer3D',
    backbone=dict(
        type='TimeSformer',
        pretrained=
        'https://download.openmmlab.com/mmaction/recognition/timesformer/vit_base_patch16_224.pth',
        num_frames=8,
        img_size=224,
        patch_size=16,
        embed_dims=768,
        in_channels=3,
        dropout_ratio=0.0,
        transformer_layers=None,
        attention_type='divided_space_time',
        norm_cfg=dict(type='LN', eps=1e-06)),
    cls_head=dict(type='TimeSformerHead', num_classes=101, in_channels=768),
    train_cfg=None,
    test_cfg=dict(average_clips='prob'))
dataset_type = 'VideoDataset'
data_root = 'Z:/Xavi/UCF101/videos'


KeyboardInterrupt: 

To save videos_small a posteriori:

In [ ]:
# progress = tqdm(range(len(datasets)*2*30))

# # Iterate over datasets
# for dataset in datasets:
    
#     # Create folder
#     if not os.path.exists(os.path.join(root_path, dataset, 'videos_small')):
#         os.mkdir(os.path.join(root_path, dataset, 'videos_small'))

#     # Iterate over classes
#     for video_class in os.listdir(os.path.join(root_path, dataset, 'videos')):
    
#         # Create folder
#         if not os.path.exists(os.path.join(root_path, dataset, 'videos_small', video_class)):
#             os.mkdir(os.path.join(root_path, dataset, 'videos_small', video_class))

#         # Iterate over videos of a class
#         for video_name in os.listdir(os.path.join(root_path, dataset, 'videos', video_class)):

#             # Load video
#             video = load_video(os.path.join(root_path, dataset, 'videos', video_class, video_name))
#             print(os.path.join(root_path, dataset, 'videos', video_class, video_name))

#             # Resize video
#             scale_factor = 256 / min(video.shape[1:3])
#             video = resize_video(video, scale_factor, scale_factor)

#             # Crop video
#             video = center_crop_video(video, 256)
            
#             # Save video
#             save_video(video, os.path.join(os.path.join(root_path, dataset, 'videos_small', video_class, video_name)))

#             # Update progress
#             progress.update(1)
# progress.close()

## Compute and store SLIC segmentation

In [8]:
chosen_datasets = ['EtriActivity3D', 'UCF101', 'Kinetics400']

In [14]:
progress = tqdm(range(len(datasets)*N_VIDEOS*2))

pipeline1 = Compose([Config.fromfile(configs['TimeSformer']).data.test.pipeline[1]])
pipeline2 = Compose([Config.fromfile(configs['TPN']).data.test.pipeline[1]])
pipeline3 = None

# Iterate over datasets
for dataset in chosen_datasets:

    # Create folder for a model and a dataset
    if not os.path.exists(os.path.join(root_path, dataset, 'segments')):
        os.mkdir(os.path.join(root_path, dataset, 'segments'))

    # Iterate over classes
    for video_class in os.listdir(os.path.join(root_path, dataset, 'videos_small')):

        # Create class folder
        if not os.path.exists(os.path.join(root_path, dataset, 'segments', video_class)):
            os.mkdir(os.path.join(root_path, dataset, 'segments', video_class))

        # Iterate over videos of a class
        for video_name in os.listdir(os.path.join(root_path, dataset, 'videos_small', video_class)):

            for slic_size, pipeline in zip(['8x224x224', '80x256x256', '16x112x112'], [pipeline1, pipeline2, pipeline3]):
            
                zip_file = os.path.join(root_path, dataset, 'segments', video_class, video_name[:-4] + '_' + slic_size + '.zip')

                # if os.path.exists(zip_file):
                #     progress.update(1)
                #     continue

                # Load video
                video = load_video(os.path.join(root_path, dataset, 'videos_small', video_class, video_name))
                
                if pipeline is None:

                    # Resize video
                    scale_factor = 112 / min(video.shape[1:3])
                    frames_in = video[::video.shape[0]//15,:,:,:]
                    frames_in = resize_video(frames_in, scale_factor, scale_factor)

                    # Crop video
                    frames_in = center_crop_video(frames_in, 112)
                else:

                    if slic_size == '8x224x224':

                        # Resize video
                        scale_factor = 224 / min(video.shape[1:3])
                        video = resize_video(video, scale_factor, scale_factor)

                        # Crop video
                        video = center_crop_video(video, 224)

                    sorted_indices, frames_in = sample_frames(video, pipeline)

                # Segment video
                time_start = time.time()
                segmenter = SlicSegmenter(frames_in)
                segments = segmenter.segment(n_segments=200, compactness=20, spacing=[0.2, 1, 1])
                
                # Save segmentation time
                total_time = time.time() - time_start
                with open(os.path.join(root_path, dataset, 'segments', video_class, video_name[:-4] + '_' + slic_size + '_time.txt'), 'w') as f:
                    f.write(str(total_time))

                save_segmentation(segments, zip_file, compressed=True)

                # Update progress
                progress.update(1)
    
    print(dataset, time.time() - time_start)
progress.close()

  0%|          | 0/180 [00:00<?, ?it/s]

EtriActivity3D 0.4157114028930664
UCF101 0.3137087821960449
Kinetics400 0.32982349395751953


## Batch explain

In [5]:
chosen_datasets = ['EtriActivity3D', 'UCF101', 'Kinetics400']

### Proposed Methods

In [ ]:
xai = None

progress = tqdm(range(len(models)*len(chosen_datasets)*N_VIDEOS*len(pipelines)))

# Iterate over models
for model_name in models:
    
    # Iterate over datasets
    for dataset in chosen_datasets:
    
        # Create folder for a model and a dataset
        if not os.path.exists(os.path.join(root_path, dataset, 'explanations')):
            os.mkdir(os.path.join(root_path, dataset, 'explanations'))
        if not os.path.exists(os.path.join(root_path, dataset, 'data-labels')):
            os.mkdir(os.path.join(root_path, dataset, 'data-labels'))
        if not os.path.exists(os.path.join(root_path, dataset, 'explanations', model_name)):
            os.mkdir(os.path.join(root_path, dataset, 'explanations', model_name))
        if not os.path.exists(os.path.join(root_path, dataset, 'data-labels', model_name)):
            os.mkdir(os.path.join(root_path, dataset, 'data-labels', model_name))
        
        # Load model
        dataset_folder = datasets_folder[dataset]
        cfg = Config.fromfile(configs[model_name])
        prepare_config(cfg, weights[dataset][model_name], dataset, dataset_folder, None, n_classes[dataset], 1)
        if model_name == 'TPN':
            cfg.model.neck.aux_head_cfg.out_channels = n_classes[dataset]
        model = init_recognizer(cfg, weights[dataset][model_name], device='cuda:0')

        # Get the pipeline to sample frames
        sample_frames_pipeline = Compose([model.cfg.data.test.pipeline[1]])

        # Replace array to imgs in the pipeline
        cfg.data.test.pipeline[2] = {
            'type': 'CustomTransform', 
            'clip_len': model.cfg.data.test.pipeline[1]['clip_len'], 
            'frame_interval': model.cfg.data.test.pipeline[1]['frame_interval'], 
            'num_clips': model.cfg.data.test.pipeline[1]['num_clips'],}

        # Remove frame sampling from the pipeline
        _ = cfg.data.test.pipeline.pop(1)
        
        # Iterate over classes
        for video_class in os.listdir(os.path.join(root_path, dataset, 'videos')):
            
            # Create class folder
            if not os.path.exists(os.path.join(root_path, dataset, 'explanations', model_name, video_class)):
                os.mkdir(os.path.join(root_path, dataset, 'explanations', model_name, video_class))
            if not os.path.exists(os.path.join(root_path, dataset, 'data-labels', model_name, video_class)):
                os.mkdir(os.path.join(root_path, dataset, 'data-labels', model_name, video_class))
                         
            # Iterate over videos of a class
            for video_name in os.listdir(os.path.join(root_path, dataset, 'videos_small', video_class)):
                
                print('Explaining: ', os.path.join(root_path, dataset, 'videos_small', video_class, video_name))
                
                # Create video folder
                if not os.path.exists(os.path.join(root_path, dataset, 'explanations', model_name, video_class, video_name)):
                    os.mkdir(os.path.join(root_path, dataset, 'explanations', model_name, video_class, video_name))
                if not os.path.exists(os.path.join(root_path, dataset, 'data-labels', model_name, video_class, video_name)):
                    os.mkdir(os.path.join(root_path, dataset, 'data-labels', model_name, video_class, video_name))

                # Check if all pipelines are done and skip video if so
                # all_done = True
                # for p_name in pipelines_names:
                #     if not os.path.exists(os.path.join(root_path, dataset, 'explanations', model_name, video_class, video_name, p_name + '.mp4')):
                #         all_done = False
                #         break
                # if all_done:
                #     progress.update(len(pipelines))
                #     continue
                
                # Load, resize and crop video
                video = load_video(os.path.join(root_path, dataset, 'videos_small', video_class, video_name))
                or_video = video
                
                # TimeSformer uses 224x224
                if model_name == 'TimeSformer':
                    scale_factor = 224 / min(video.shape[1:3])
                    video = resize_video(video, scale_factor, scale_factor)
                    video = center_crop_video(video, 224)

                for pipeline, p_name in zip(pipelines, pipelines_names):

                    if os.path.exists(os.path.join(root_path, dataset, 'explanations', model_name, video_class, video_name, p_name + '.mp4')):
                        progress.update(1)
                        continue

                    # Get list with indexes of frames
                    sorted_indices, frames_in = sample_frames(video, sample_frames_pipeline)
                        
                    # Run pipeline. SLIC is loaded instead of computed
                    start_time = time.time()
                    xai = Xai(frames_in, partial(batch_predict, sorted_indices=sorted_indices), pipeline)
                    if p_name in slic_methods:
                        if model_name == 'TimeSformer':
                            slic_size = '8x224x224'
                        else:
                            slic_size = '80x256x256'
                        xai.segments = load_segmentation(os.path.join(root_path, dataset, 'segments', video_class, video_name[:-4] + '_' + slic_size + '.zip'))
                    else:
                        xai.segment()
                    xai.perturb()
                    xai.explain()
                    xai.video = or_video # Visualize on the original video (not on the sample one).
                    xai.visualize()
                    end_time = time.time()
                    
                    # Save results
                    save_video(
                        xai.exp_vid, 
                        os.path.join(root_path, dataset, 'explanations', model_name, video_class, video_name, p_name + '.mp4'))
                    np.save(os.path.join(root_path, dataset, 'data-labels', model_name, video_class, video_name, p_name+'-data.npy'), xai.data)
                    np.save(os.path.join(root_path, dataset, 'data-labels', model_name, video_class, video_name, p_name+'-labels.npy'), xai.labels)
                    np.save(os.path.join(root_path, dataset, 'data-labels', model_name, video_class, video_name, p_name+'-scores.npy'), xai.scores)
                    
                    with open(os.path.join(root_path, dataset, 'data-labels', model_name, video_class, video_name, p_name+'-time.txt'), "w") as f:
                        f.write(str(end_time - start_time))
                
                    # Update progress
                    progress.update(1)
progress.close()

#### For speed comparison only

In [ ]:
xai = None

progress = tqdm(range(len(models)*len(chosen_datasets)*N_VIDEOS*len(pipelines)))

# Iterate over models
for model_name in models:
    
    # Iterate over datasets
    for dataset in chosen_datasets:
    
        # Create folder for a model and a dataset
        if not os.path.exists(os.path.join(root_path, dataset, 'explanations')):
            os.mkdir(os.path.join(root_path, dataset, 'explanations'))
        if not os.path.exists(os.path.join(root_path, dataset, 'data-labels')):
            os.mkdir(os.path.join(root_path, dataset, 'data-labels'))
        if not os.path.exists(os.path.join(root_path, dataset, 'explanations', model_name)):
            os.mkdir(os.path.join(root_path, dataset, 'explanations', model_name))
        if not os.path.exists(os.path.join(root_path, dataset, 'data-labels', model_name)):
            os.mkdir(os.path.join(root_path, dataset, 'data-labels', model_name))
        
        # Load model
        dataset_folder = datasets_folder[dataset]
        cfg = Config.fromfile(configs[model_name])
        prepare_config(cfg, weights[dataset][model_name], dataset, dataset_folder, None, n_classes[dataset], 1)
        if model_name == 'TPN':
            cfg.model.neck.aux_head_cfg.out_channels = n_classes[dataset]
        model = init_recognizer(cfg, weights[dataset][model_name], device='cuda:0')

        # Get the pipeline to sample frames
        sample_frames_pipeline = Compose([model.cfg.data.test.pipeline[1]])

        # Replace array to imgs in the pipeline
        cfg.data.test.pipeline[2] = {
            'type': 'CustomTransform', 
            'clip_len': model.cfg.data.test.pipeline[1]['clip_len'], 
            'frame_interval': model.cfg.data.test.pipeline[1]['frame_interval'], 
            'num_clips': model.cfg.data.test.pipeline[1]['num_clips'],}

        # Remove frame sampling from the pipeline
        _ = cfg.data.test.pipeline.pop(1)
        
        # Iterate over classes
        for video_class in os.listdir(os.path.join(root_path, dataset, 'videos')):
            
            # Create class folder
            if not os.path.exists(os.path.join(root_path, dataset, 'explanations', model_name, video_class)):
                os.mkdir(os.path.join(root_path, dataset, 'explanations', model_name, video_class))
            if not os.path.exists(os.path.join(root_path, dataset, 'data-labels', model_name, video_class)):
                os.mkdir(os.path.join(root_path, dataset, 'data-labels', model_name, video_class))
                         
            # Iterate over videos of a class
            for video_name in os.listdir(os.path.join(root_path, dataset, 'videos_small', video_class)):
                
                print('Explaining: ', os.path.join(root_path, dataset, 'videos_small', video_class, video_name))
                
                # Create video folder
                if not os.path.exists(os.path.join(root_path, dataset, 'explanations', model_name, video_class, video_name)):
                    os.mkdir(os.path.join(root_path, dataset, 'explanations', model_name, video_class, video_name))
                if not os.path.exists(os.path.join(root_path, dataset, 'data-labels', model_name, video_class, video_name)):
                    os.mkdir(os.path.join(root_path, dataset, 'data-labels', model_name, video_class, video_name))

                # Check if all pipelines are done and skip video if so
                # all_done = True
                # for p_name in pipelines_names:
                #     if not os.path.exists(os.path.join(root_path, dataset, 'explanations', model_name, video_class, video_name, p_name + '.mp4')):
                #         all_done = False
                #         break
                # if all_done:
                #     progress.update(len(pipelines))
                #     continue
                
                # Load, resize and crop video
                video = load_video(os.path.join(root_path, dataset, 'videos_small', video_class, video_name))
                or_video = video
                
                scale_factor = 112 / min(video.shape[1:3])
                video = video[::video.shape[0]//15,:,:,:]
                video = resize_video(video, scale_factor, scale_factor)
                video = center_crop_video(video, 112)

                for pipeline, p_name in zip(pipelines, pipelines_names):

                    if os.path.exists(os.path.join(root_path, dataset, 'data-labels', model_name, video_class, video_name, p_name+'-time-16x112x112.txt')):
                        progress.update(1)
                        continue

                    # Get list with indexes of frames
                    sorted_indices, frames_in = sample_frames(video, sample_frames_pipeline)
                        
                    # Run pipeline. SLIC is loaded instead of computed
                    start_time = time.time()
                    xai = Xai(frames_in, partial(batch_predict, sorted_indices=sorted_indices), pipeline)
                    if p_name in slic_methods:
                        slic_size = '16x112x112'
                        xai.segments = load_segmentation(os.path.join(root_path, dataset, 'segments', video_class, video_name[:-4] + '_' + slic_size + '.zip'))
                    else:
                        xai.segment()
                    xai.perturb()
                    xai.explain()
                    xai.video = or_video # Visualize on the original video (not on the sample one).
                    xai.visualize()
                    end_time = time.time()
                    
                    # Save results
                    with open(os.path.join(root_path, dataset, 'data-labels', model_name, video_class, video_name, p_name+'-time-16x112x112.txt'), "w") as f:
                        f.write(str(end_time - start_time))
                
                    # Update progress
                    progress.update(1)
progress.close()

### Other methods

In [8]:
# methods_names = ['SaliencyTubes', 'GradCAM', 'AOSA', 'AOSA_SGL', '3D_EP', 'STEP']
# methods_names = ['SaliencyTubes', 'GradCAM', 'AOSA', 'AOSA_SGL']
# methods_names = ['AOSA', 'AOSA_SGL']
methods_names = ['SaliencyTubes', 'GradCAM']
# methods_names = ['3D_EP', 'STEP']

# methods = [
#     lambda conv_layer, out_layer: SaliencyTubes(conv_layer, out_layer, classifier_fn1, 3),
#     lambda model, conv_layer: GradCAM(model, preprocess_fn, conv_layer, 3),
#     lambda: AOSA(classifier_fn2, N_stack_mask=3),
#     lambda: AOSA(classifier_fn2, N_stack_mask=1),
#     lambda: EP(classifier_fn3, method='3d_ep'),
#     lambda: EP(classifier_fn3, method='step'),
# ]
# methods = [
#     lambda conv_layer, out_layer: SaliencyTubes(conv_layer, out_layer, classifier_fn1, 3),
#     lambda model, conv_layer: GradCAM(model, preprocess_fn, conv_layer, 3),
#     lambda sorted_indices: AOSA(partial(classifier_fn4, sorted_indices=sorted_indices), N_stack_mask=3),
#     lambda sorted_indices: AOSA(partial(classifier_fn4, sorted_indices=sorted_indices), N_stack_mask=1),
# ]
# methods = [
#     lambda: AOSA(classifier_fn2, N_stack_mask=3),
#     lambda: AOSA(classifier_fn2, N_stack_mask=1),
# ]
methods = [
    lambda conv_layer, out_layer: SaliencyTubes(conv_layer, out_layer, classifier_fn1, 3),
    lambda model, conv_layer: GradCAM(model, preprocess_fn, conv_layer, 3),
]
# methods = {
#     'AOSA': lambda sorted_indices: AOSA(partial(classifier_fn4, sorted_indices=sorted_indices), N_stack_mask=3, downscale=False),
#     'AOSA_SGL': lambda sorted_indices: AOSA(partial(classifier_fn4, sorted_indices=sorted_indices), N_stack_mask=1, downscale=False),
# }
# methods = [
#     lambda: EP(classifier_fn3, method='3d_ep'),
#     lambda: EP(classifier_fn3, method='step'),
# ]

def classifier_fn1(video):
    inference_recognizer(model, video)

def classifier_fn2(videos):
    results = []
    for video in videos:
        results.append(inference_recognizer(model, video.cpu().detach().squeeze().numpy().transpose(1, 2, 3, 0)))
    return np.array(results)

def classifier_fn3(videos):
    results = []
    for video in videos:
        results.append(inference_recognizer(model, video.cpu().detach().squeeze().numpy().transpose(1, 2, 3, 0)))
    return torch.tensor(np.array(results))

def classifier_fn4(videos, sorted_indices):
    results = []
    for video in videos:
        video = video.cpu().detach().squeeze().numpy().transpose(1, 2, 3, 0)
        frames_out = unsort_frames(sorted_indices, video)
        results.append(inference_recognizer(model, frames_out))
    return np.array(results)

def preprocess_fn(video):
    cfg = model.cfg
    device = next(model.parameters()).device  # model device
    test_pipeline = cfg.data.test.pipeline
    modality_map = {2: 'Flow', 3: 'RGB'}
    modality = modality_map.get(video.shape[-1])
    data = dict(
        total_frames=video.shape[0],
        label=-1,
        start_index=0,
        array=video,
        modality=modality)
    for i in range(len(test_pipeline)):
        if 'Decode' in test_pipeline[i]['type']:
            test_pipeline[i] = dict(type='ArrayDecode')
    test_pipeline = [x for x in test_pipeline if 'Init' not in x['type']]

    test_pipeline = Compose(test_pipeline)
    data = test_pipeline(data)
    data = collate([data], samples_per_gpu=1)

    if next(model.parameters()).is_cuda:
        # scatter to specified GPU
        data = scatter(data, [device])[0]

    # data['inputs'] = data['imgs']
    data = {'inputs': data['imgs'], 'label': data['label']}
    
    return data

In [9]:
progress = tqdm(range(len(models)*len(chosen_datasets)*N_VIDEOS*len(methods)))

# Iterate over models
for model_name in models:
    
    # Iterate over datasets
    for dataset in chosen_datasets:
        
        # Load model
        dataset_folder = datasets_folder[dataset]
        cfg = Config.fromfile(configs[model_name])
        prepare_config(cfg, weights[dataset][model_name], dataset, dataset_folder, None, n_classes[dataset], 1)
        if model_name == 'TPN':
            cfg.model.neck.aux_head_cfg.out_channels = n_classes[dataset]
        model = init_recognizer(cfg, weights[dataset][model_name], device='cuda:0')
        
        # Iterate over classes
        for video_class in os.listdir(os.path.join(root_path, dataset, 'videos')):
                         
            # Iterate over videos of a class
            for video_name in os.listdir(os.path.join(root_path, dataset, 'videos', video_class)):
                print('Explaining: ', os.path.join(root_path, dataset, 'videos', video_class, video_name))
                
                # Iterate over methods
                for method, method_name in zip(methods, methods_names):
                    
                    # # Skip if explanation already exists
                    # if os.path.exists(os.path.join(root_path, dataset, 'data-labels', model_name, video_class, video_name, method_name+'-time.txt')):
                    #     progress.update(1)
                    #     continue
                    
                    # Load, resize and crop video
                    video = load_video(os.path.join(root_path, dataset, 'videos_small', video_class, video_name))
                    # scale_factor = 256 / min(video.shape[1:3])
                    # video = resize_video(video, scale_factor, scale_factor)
                    # video = center_crop_video(video, 256)
                    
                    # Choose explainer
                    if method_name == 'SaliencyTubes':
                        if model_name == 'TPN':
                            explainer = method(conv_layer=model.neck.pyramid_fusion, out_layer=model.cls_head.fc_cls)
                        elif model_name == 'TANet':
                            explainer = method(conv_layer=model.backbone, out_layer=model.cls_head.fc_cls)
                        else:
                            continue
                    elif method_name == 'GradCAM':
                        if model_name == 'TPN':
                            explainer = method(model, conv_layer='neck/pyramid_fusion')
                        elif model_name == 'TANet':
                            explainer = method(model, conv_layer='backbone')
                        else:
                            continue
                    else:
                        explainer = method()
                    
                    # Explain and visualize
                    start_time = time.time()
                    score_map = explainer.explain(video, int(video_class))
                    visualizer = HeatmapVisualizer(video, None, None)
                    rgb_score_map = visualizer.visualize(score_map)
                    exp_vid = visualizer.visualize_on_video(rgb_score_map)
                    end_time = time.time()
                    
                # Save results
                    save_video(
                        exp_vid, 
                        os.path.join(root_path, dataset, 'explanations', model_name, video_class, video_name, method_name + '.mp4'))
                    np.save(os.path.join(root_path, dataset, 'data-labels', model_name, video_class, video_name, method_name+'-scores.npy'), score_map)
                    
                    with open(os.path.join(root_path, dataset, 'data-labels', model_name, video_class, video_name, method_name+'-time.txt'), "w") as f:
                        f.write(str(end_time - start_time))

                    # Clear memory
                    torch.cuda.empty_cache()

                    # Update progress
                    progress.update(1)
progress.close()

  0%|          | 0/540 [00:00<?, ?it/s]

load checkpoint from local path: Z:/Xavi/EtriActivity3D/training/TimeSformer/1/epoch_10.pth
Explaining:  Z:/Xavi/VideoXAI/batch_explanations\EtriActivity3D\videos\0\A001_P046_G002_C001.mp4
Explaining:  Z:/Xavi/VideoXAI/batch_explanations\EtriActivity3D\videos\1\A002_P065_G001_C001.mp4
Explaining:  Z:/Xavi/VideoXAI/batch_explanations\EtriActivity3D\videos\1\A002_P096_G003_C004.mp4
Explaining:  Z:/Xavi/VideoXAI/batch_explanations\EtriActivity3D\videos\13\A014_P015_G003_C001.mp4
Explaining:  Z:/Xavi/VideoXAI/batch_explanations\EtriActivity3D\videos\14\A015_P013_G002_C008.mp4
Explaining:  Z:/Xavi/VideoXAI/batch_explanations\EtriActivity3D\videos\14\A015_P035_G002_C004.mp4
Explaining:  Z:/Xavi/VideoXAI/batch_explanations\EtriActivity3D\videos\15\A016_P021_G002_C006.mp4
Explaining:  Z:/Xavi/VideoXAI/batch_explanations\EtriActivity3D\videos\15\A016_P021_G003_C008.mp4
Explaining:  Z:/Xavi/VideoXAI/batch_explanations\EtriActivity3D\videos\16\A017_P063_G002_C002.mp4
Explaining:  Z:/Xavi/VideoXAI

KeyboardInterrupt: 

## Copy explanations

In [11]:
chosen_pipelines = ['3D-LIME-NEW', '3D-Kernel-SHAP-NEW', 'LV-LOCO-NEW', 'LV-Univ-Pred-NEW', '3D-RISE-NEW', '3D-Sampled-Occl-Sens-NEW', 'AOSA', 'AOSA_SGL', 'SaliencyTubes', 'GradCAM']
chosen_datasets = ['EtriActivity3D', 'UCF101', 'Kinetics400']
progress = tqdm(range(len(models)*len(chosen_datasets)*N_VIDEOS*len(chosen_pipelines)))
dest_path = 'Z:/Xavi/VideoXAI/saved_explanations'

if not os.path.exists(dest_path):
    os.mkdir(dest_path)
    
# Iterate over datasets
for dataset in chosen_datasets:

    if not os.path.exists(os.path.join(dest_path, dataset)):
        os.mkdir(os.path.join(dest_path, dataset))
    if not os.path.exists(os.path.join(dest_path, dataset, 'explanations')):
        os.mkdir(os.path.join(dest_path, dataset, 'explanations'))
    if not os.path.exists(os.path.join(dest_path, dataset, 'videos')):
        os.mkdir(os.path.join(dest_path, dataset, 'videos'))

    # Iterate over models
    for model_name in models:

        if not os.path.exists(os.path.join(dest_path, dataset, 'explanations', model_name)):
            os.mkdir(os.path.join(dest_path, dataset, 'explanations', model_name))
        
        # Iterate over classes
        for video_class in os.listdir(os.path.join(root_path, dataset, 'videos')):

            if not os.path.exists(os.path.join(dest_path, dataset, 'explanations', model_name, video_class)):
                os.mkdir(os.path.join(dest_path, dataset, 'explanations', model_name, video_class))
            if not os.path.exists(os.path.join(dest_path, dataset, 'videos', video_class)):
                os.mkdir(os.path.join(dest_path, dataset, 'videos', video_class))
                         
            # Iterate over videos of a class
            for video_name in os.listdir(os.path.join(root_path, dataset, 'videos_small', video_class)):

                if not os.path.exists(os.path.join(dest_path, dataset, 'explanations', model_name, video_class, video_name)):
                    os.mkdir(os.path.join(dest_path, dataset, 'explanations', model_name, video_class, video_name))

                # Copy video
                vid_path = os.path.join(root_path, dataset, 'videos_small', video_class, video_name)
                vid_path_dest = os.path.join(dest_path, dataset, 'videos', video_class, video_name)
                shutil.copyfile(vid_path, vid_path_dest)

                for p_name in chosen_pipelines:

                    if model_name == 'TimeSformer' and p_name in ['GradCAM', 'SaliencyTubes']:
                        progress.update(1)
                        continue
                    
                    # Copy explanation
                    exp_vid_path = os.path.join(root_path, dataset, 'explanations', model_name, video_class, video_name, p_name + '.mp4')
                    if 'NEW' in p_name:
                        p_name = p_name.replace('-NEW', '')
                    exp_vid_path_dest = os.path.join(dest_path, dataset, 'explanations', model_name, video_class, video_name, p_name + '.mp4')
                    shutil.copyfile(exp_vid_path, exp_vid_path_dest)
                
                    # Update progress
                    progress.update(1)
progress.close()

  0%|          | 0/2700 [00:00<?, ?it/s]

## Fix explanations

In [37]:
progress = tqdm(range(len(models)*len(datasets)*N_VIDEOS))
# Iterate over models
for model_name in models:
    
    # Iterate over datasets
    for dataset in datasets:
        
        # Load model
#         model = init_recognizer(configs[dataset][model_name], weights[dataset][model_name], device='cuda:0')
        
        # Iterate over classes
        for video_class in os.listdir(os.path.join(root_path, dataset, 'videos')):
                         
            # Iterate over videos of a class
            for video_name in os.listdir(os.path.join(root_path, dataset, 'videos', video_class)):
                
#                # To remove files
#                 os.remove(os.path.join(root_path, dataset, 'explanations', model_name, video_class, video_name, 'OCCLUSION_EXP.mp4'))
#                 os.remove(os.path.join(root_path, dataset, 'data-labels', model_name, video_class, video_name, 'OCCLUSION_EXP-time.txt'))
                    
#                 # Load video
#                 video = load_video(os.path.join(root_path, dataset, 'videos_small', video_class, video_name))
#                 print(os.path.join(root_path, dataset, 'videos_small', video_class, video_name))

#                # Load segments
#                 segments = load_segmentation(os.path.join(root_path, dataset, 'segments', video_class, video_name[:-3] + 'zip'), compressed=True)
                
#                 for xai_method in pipelines_names:
                    
#                     # Load scores
#                     scores = np.load(os.path.join(root_path, dataset, 'data-labels', model_name, video_class, video_name, xai_method+'-scores.npy'))
                                     
#                     # Get score_map
#                     visualizer = HeatmapVisualizer(video, segments, scores)
# #                     visualizer = PosNegVisualizer(video, segments, scores)
#                     score_map = visualizer.get_score_map(min_accum=0.3, improve_background=True)
#                     exp_vid = visualizer.visualize_on_video(visualizer.visualize(score_map, improve_background=True))
#                     save_video(
#                         exp_vid, 
#                         os.path.join(root_path, dataset, 'explanations', model_name, video_class, video_name, xai_method + '.mp4'))

                # Update progress
                progress.update(1)
progress.close()

A Jupyter Widget

In [13]:
progress.close()